In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers, Sequential, models
import pathlib
from tensorflow.keras import regularizers


"""### Set all the Constants"""


BATCH_SIZE = 32
IMAGE_SIZE = 256
CHANNELS=3
EPOCHS=400


from tensorflow.keras.layers import Lambda


# Bulanıklık fonksiyonu
def apply_blur(image):
   # Görüntüyü bulanıklaştırma işlemi
   blurred_image = cv2.GaussianBlur(image.numpy(), (5,5), 0)
   return blurred_image


data_augmentation = tf.keras.Sequential([
  # layers.RandomFlip("horizontal_and_vertical"),
  # layers.RandomRotation(0.2),
   Lambda(lambda x: apply_blur(x)),  # Bulanıklık uygulama
   layers.GaussianNoise(0.1),
   layers.RandomContrast(0.2),  # Rastgele kontrast değişimi
   layers.RandomBrightness(0.2),  # Rastgele parlaklık değişimi
   layers.RandomZoom(0.2, 0.2),  # Rastgele yakınlaştırma
   layers.RandomTranslation(0.2, 0.2),  # Rastgele çeviri
  # layers.RandomAffine(0.2),  # Rastgele kaydırma
])




"""### Import data into tensorflow dataset object"""


dataset = tf.keras.preprocessing.image_dataset_from_directory(
   "/home/han/Downloads/train",
   seed=12,
   shuffle=True,
   image_size=(IMAGE_SIZE,IMAGE_SIZE),
   batch_size=BATCH_SIZE
)


class_names = dataset.class_names
class_names




"""**Train, Test, validation data split**"""


def get_dataset_partitions_tf(ds, train_split=0.8, val_split=0.1, test_split=0.1, shuffle=True, shuffle_size=10000):
   assert (train_split + test_split + val_split) == 1


   ds_size = len(ds)


   if shuffle:
       ds = ds.shuffle(shuffle_size, seed=12)


   train_size = int(train_split * ds_size)
   val_size = int(val_split * ds_size)


   train_ds = ds.take(train_size)
   val_ds = ds.skip(train_size).take(val_size)
   test_ds = ds.skip(train_size).skip(val_size)


   return train_ds, val_ds, test_ds


train_ds, val_ds, test_ds = get_dataset_partitions_tf(dataset)


print("Size of Data is :{0} \nBatch size of Training Data is :{1}\nBatch size of Validation Data is :{2} \nBatch size of Testing Data is :{3} " .format(len(dataset), len(train_ds), len(val_ds), len(test_ds)))


"""### Cache, Shuffle, and Prefetch the Dataset"""


train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)


"""## Building the Model


### Creating a Layer for Resizing and Normalization
Before we feed our images to network, we should be resizing it to the desired size.
Moreover, to improve model performance, we should normalize the image pixel value (keeping them in range 0 and 1 by dividing by 256).
This should happen while training as well as inference. Hence we can add that as a layer in our Sequential Model.


You might be thinking why do we need to resize (256,256) image to again (256,256). You are right we don't need to but this will be useful when we are done with the training and start using the model for predictions. At that time somone can supply an image that is not (256,256) and this layer will resize it
"""


resize_and_rescale = tf.keras.Sequential([
 layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
 layers.Rescaling(1./255),
])


"""### Model Architecture
We use a CNN coupled with a Softmax activation in the output layer. We also add the initial layers for resizing, normalization and Data Augmentation.
"""


input_shape = (BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
n_classes = 5


model = models.Sequential([
   resize_and_rescale,
   layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
   layers.MaxPooling2D((2, 2)),
   layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
   layers.MaxPooling2D((2, 2)),
   layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
   layers.MaxPooling2D((2, 2)),
   layers.Conv2D(256, kernel_size=(3, 3), activation='relu'),
   layers.MaxPooling2D((2, 2)),
   layers.Conv2D(512, kernel_size=(3, 3), activation='relu'),
   layers.MaxPooling2D((2, 2)),
   layers.Flatten(),
   layers.Dropout(0.7),
   layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
   layers.Dropout(0.8),  # Dropout for regularization
   layers.Dense(n_classes, activation='softmax'),


])


model.build(input_shape=input_shape)


model.summary()


"""### Compiling the Model
We use `adam` Optimizer, `SparseCategoricalCrossentropy` for losses, `accuracy` as a metric
"""


# Compile the model with a lower learning rate
model.compile(
   optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
   loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
   metrics=['accuracy']
)


# Early stopping tanımlama
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)


history = model.fit(
   train_ds,
   batch_size=BATCH_SIZE,
   validation_data=val_ds,
   verbose=1,
   epochs=EPOCHS,
   callbacks=[early_stopping],
)


scores = model.evaluate(test_ds)




# Plotting the Accuracy and Loss Curves
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']


plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.title('Training and Validation Accuracy')


plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.title('Training and Validation Loss')
plt.show()




model.save("/home/han/Documents/EE470_Model_BEST_5_.h5")
